In [23]:
%pip install statsmodels ucimlrepo seaborn pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import statsmodels.api as sm
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [29]:
# fetch dataset
automobile = fetch_ucirepo(id=10)

# data (as pandas dataframes)
X = automobile.data.features
y = automobile.data.targets

# replace missing values with the average of each column over the whole dataset
print("Missing values before imputation:")
print(X.isna().sum()[X.isna().sum() > 0])

fill_values = X.mean(numeric_only=True)
# num-of-doors is either 2 or 4, so use the most common value instead of an average like 3.1
fill_values['num-of-doors'] = X['num-of-doors'].mode()[0]
X = X.fillna(fill_values)

print("Missing values after imputation:", X.isna().sum().sum())

# one-hot encode the categorical (text) columns, then add the intercept
X = pd.get_dummies(X, drop_first=True, dtype=float)
X = sm.add_constant(X)

# split x and y into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

Missing values before imputation:
price                 4
peak-rpm              2
horsepower            2
stroke                4
bore                  4
num-of-doors          2
normalized-losses    41
dtype: int64
Missing values after imputation: 0
(164, 61) (41, 61)


In [30]:
# Fit OLS on the encoded training data
ols = sm.OLS(y_train, X_train).fit()
print(ols.summary())

# Evaluate on the test set
y_pred = ols.predict(X_test)
print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R^2:", r2_score(y_test, y_pred))

                            OLS Regression Results                            
Dep. Variable:              symboling   R-squared:                       0.855
Model:                            OLS   Adj. R-squared:                  0.776
Method:                 Least Squares   F-statistic:                     10.93
Date:                Sat, 19 Sep 2026   Prob (F-statistic):           1.55e-25
Time:                        18:26:02   Log-Likelihood:                -109.93
No. Observations:                 164   AIC:                             335.9
Df Residuals:                     106   BIC:                             515.6
Df Model:                          57                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    8.5906 